# Lab 3: PCA și K-means Clustering (Ames Housing)

Grupăm casele după caracteristicile lor, fără să folosim prețul ca intrare.
**PCA** reduce numărul de dimensiuni, iar **K-means** caută grupuri de case
asemănătoare în spațiul redus. Continuăm curățarea din L1/L2 și rezolvăm
cerințele obligatorii, plus bonusul cu grafice 3D și comparația cu clasele de preț.

Rulăm celulele în ordine, cu **Restart Kernel → Run All**. Datele sunt deja în
`../data/AmesHousing.csv`. Calea funcționează și dacă pornim din rădăcina repo-ului.
Instalare, dacă este necesară: `%pip install -r requirements.txt` din folderul L3.
Rezultatele și graficele se calculează din date; tabelele sunt exportate în
`L3/rezultate/`. Analiza este exploratorie: folosim întregul set curățat,
iar casele inventate demonstrează inferența, nu măsoară generalizarea pe un test independent.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.preprocessing import MinMaxScaler, Normalizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, adjusted_rand_score
from threadpoolctl import threadpool_limits

SEED = 42
THRESHOLDS = [0.98, 0.95, 0.90, 0.80]
K_VALUES = list(range(2, 11))
DATA_PATH = Path('../data/AmesHousing.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/AmesHousing.csv')
OUTPUT_DIR = DATA_PATH.resolve().parent.parent / 'L3' / 'rezultate'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Curățarea datelor și codificarea atributelor

Eliminăm aceiași outlieri ca în L1/L2: case cu `Gr Liv Area > 4000` și
`SalePrice < 300000`. Valorile lipsă care înseamnă absența unui element devin
`None`; cele numerice rămase sunt completate cu mediana, iar cele nominale cu moda.
`Lot Frontage` este completat mai întâi cu mediana cartierului, ca în strategia B din L2.

Calitățile ordonate primesc coduri de la 0 (absent) la 5 (excelent).
Celelalte categorii folosesc **One-Hot Encoding complet**. `MS SubClass` este
categorial, chiar dacă în CSV apare ca număr. Păstrăm toate categoriile pentru
ca o categorie de referință eliminată să nu aibă artificial alte distanțe.

Excludem `SalePrice`, clasele de preț și identificatorii `Order`, `PID` din
PCA/K-means. Prețul apare doar ulterior, pentru interpretare. Nu adăugăm
atributele sintetice din L2: repetarea suprafeței/calității le-ar mări implicit
ponderea în distanțe. Tratarea calității prin coduri presupune distanțe egale
între niveluri; este o alegere de modelare, nu o proprietate demonstrată a datelor.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('Lipseste data/AmesHousing.csv. Verifica directorul curent.')
raw = pd.read_csv(DATA_PATH, sep='\t')
raw.columns = raw.columns.str.strip()
required = {'SalePrice', 'Gr Liv Area', 'Overall Qual', 'Year Built'}
assert required.issubset(raw.columns), 'Lipsesc coloanele originale Ames.'
outliers = (raw['Gr Liv Area'] > 4000) & (raw['SalePrice'] < 300000)
df = raw.loc[~outliers].reset_index(drop=True).copy()
X_raw = df.drop(columns=['SalePrice', 'Order', 'PID'], errors='ignore')
print(f'{len(df)} case; {outliers.sum()} outlieri eliminati; {X_raw.shape[1]} atribute initiale.')

ABSENCE = ['Alley', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1',
           'BsmtFin Type 2', 'Fireplace Qu', 'Garage Type', 'Garage Finish',
           'Garage Qual', 'Garage Cond', 'Pool QC', 'Fence', 'Misc Feature', 'Mas Vnr Type']
QUALITY = ['Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond', 'Heating QC',
           'Kitchen Qual', 'Fireplace Qu', 'Garage Qual', 'Garage Cond', 'Pool QC']

# Pastram medianele invatate pentru transformarea caselor noi.
frontage_by_neighborhood = X_raw.groupby('Neighborhood')['Lot Frontage'].median()
frontage_median = X_raw['Lot Frontage'].median()

def clean_features(frame):
    x = frame.copy()
    x['Lot Frontage'] = x['Lot Frontage'].fillna(
        x['Neighborhood'].map(frontage_by_neighborhood)
    ).fillna(frontage_median)
    for col in ABSENCE:
        x[col] = x[col].fillna('None')
    for col in QUALITY:
        x[col] = x[col].map({'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5})
    x['MS SubClass'] = x['MS SubClass'].astype(str)
    return x

X = clean_features(X_raw)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in X if col not in num_cols]

# Logaritmam doar masuratori nenegative cu asimetrie pronuntata spre dreapta.
log_candidates = ['Lot Frontage', 'Lot Area', 'Mas Vnr Area', 'BsmtFin SF 1',
                  'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF',
                  '2nd Flr SF', 'Gr Liv Area', 'Garage Area', 'Wood Deck SF',
                  'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch',
                  'Pool Area', 'Misc Val']
LOG_COLS = [col for col in log_candidates if X[col].min() >= 0 and X[col].skew() > 1]
print('Atribute transformate cu log1p:', LOG_COLS)

def make_preprocessor():
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), num_cols),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ], verbose_feature_names_out=False)

def before_encoding(frame, use_log):
    x = clean_features(frame)
    if use_log:
        x[LOG_COLS] = np.log1p(x[LOG_COLS])
    return x


## 2. PCA: 98%, 95%, 90% și 80% din varianță

Comparăm patru variante: **min-max**, **normalizare L2**, și fiecare dintre ele
cu **log1p** înaintea scalării. Min-max scalează fiecare coloană în intervalul
$[0,1]$; normalizarea L2 împarte fiecare rând la lungimea sa euclidiană.
Normalizarea nu înseamnă standardizare cu medie 0 și deviație standard 1.
Aplicăm L2 direct datelor codificate pentru a vedea și efectul unităților:
suprafețele și anii pot domina față de indicatorii binari.

PCA centrează datele, dar nu le scalează automat. Antrenăm un PCA complet
pentru fiecare variantă și păstrăm primul număr de componente care atinge
pragul cerut. Este echivalent cu trunchierea aceluiași PCA, fără `whiten`.
Cele 98% reprezintă varianță păstrată, nu acuratețe de predicție.
Tabelul complet `explained_variance_ratio_` este afișat și exportat pentru fiecare variantă.


In [ ]:
experiments = {}
variance_rows = []
variance_details = []
for use_log in [False, True]:
    for scaling in ['minmax', 'normalize']:
        name = scaling + (' + log1p' if use_log else '')
        prep = make_preprocessor()
        encoded = prep.fit_transform(before_encoding(X_raw, use_log))
        scaler = MinMaxScaler() if scaling == 'minmax' else Normalizer(norm='l2')
        scaled = scaler.fit_transform(encoded)
        with threadpool_limits(limits=2):
            pca = PCA(svd_solver='full')
            scores = pca.fit_transform(scaled)
        cumulative = np.cumsum(pca.explained_variance_ratio_)
        counts = {t: int(np.searchsorted(cumulative, t) + 1) for t in THRESHOLDS}
        experiments[name] = dict(prep=prep, scaler=scaler, pca=pca, scores=scores,
                                 counts=counts, use_log=use_log)
        for t, count in counts.items():
            variance_rows.append([name, t, encoded.shape[1], count, cumulative[count-1]])
        detail = pd.DataFrame({'varianta': name, 'PC': np.arange(1, len(cumulative)+1),
                               'varianta_explicata': pca.explained_variance_ratio_,
                               'varianta_cumulata': cumulative})
        variance_details.append(detail)
        with pd.option_context('display.max_rows', None):
            display(detail)
        plt.plot(detail['PC'], cumulative, label=name)
for t in THRESHOLDS:
    plt.axhline(t, color='gray', linestyle=':', alpha=.5)
plt.xlabel('Numarul componentelor principale')
plt.ylabel('Proportia cumulata a variantei explicate')
plt.legend()
plt.tight_layout()
plt.show()
variance_table = pd.DataFrame(variance_rows, columns=[
    'varianta', 'prag', 'atribute_codificate', 'componente', 'varianta_pastrata'])
display(variance_table)
pd.concat(variance_details).to_csv(OUTPUT_DIR / 'varianta_per_componenta.csv', index=False)
variance_table.to_csv(OUTPUT_DIR / 'rezumat_pca.csv', index=False)


## 3. Alegerea lui k: metoda cotului și analiza silhouette

Pentru fiecare combinație de preprocesare și prag PCA antrenăm K-means
cu `k=1,...,10`, inițializare `k-means++`, 10 porniri și seed fix.
**Inerția** este suma distanțelor pătratice până la centroizi; ea scade
odată cu k. Căutăm zona după care îmbunătățirile devin mai mici.

Pentru o alegere reproductibilă estimăm cotul prin distanța maximă dintre
curba inerției normalizate și segmentul care unește capetele ei.
Este o euristică pe intervalul testat, nu o dovadă că există un k optim unic.
Folosim acest k pentru graficele și profilurile finale; afișăm separat k cu
silhouette maxim. Dacă cele două criterii diferă, comparăm separarea,
dimensiunea și interpretabilitatea grupurilor, nu ascundem diferența.
`K_OVERRIDES` permite o alegere manuală după examinarea graficelor.

Silhouette se calculează în **toate componentele păstrate**, pentru fiecare casă:
aproape de 1 înseamnă separare bună, aproape de 0 înseamnă graniță între
grupuri, iar o valoare negativă poate indica o atribuire nepotrivită.
Nu comparăm direct inerțiile din spații cu scalări diferite.


In [ ]:
K_OVERRIDES = {}  # Exemplu: {('minmax', 0.98): 3}

def estimate_elbow(inertias):
    x = np.linspace(0, 1, len(inertias))
    y = (np.asarray(inertias) - inertias[-1]) / (inertias[0] - inertias[-1])
    distances = (1 - x) - y
    return int(np.argmax(distances[1:-1]) + 2)

results = {}
comparison_rows = []
metric_rows = []
with threadpool_limits(limits=2):
    for name, exp in experiments.items():
        fig, axes = plt.subplots(2, 4, figsize=(18, 8))
        for col, t in enumerate(THRESHOLDS):
            z = exp['scores'][:, :exp['counts'][t]]
            models = {}
            inertias = []
            silhouettes = {}
            for k in [1] + K_VALUES:
                model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(z)
                models[k] = model
                inertias.append(model.inertia_)
                if k > 1:
                    silhouettes[k] = silhouette_samples(z, model.labels_)
                metric_rows.append([name, t, k, model.inertia_,
                                    silhouettes[k].mean() if k > 1 else np.nan])
            elbow_k = estimate_elbow(inertias)
            silhouette_k = max(K_VALUES, key=lambda k: silhouettes[k].mean())
            best_k = K_OVERRIDES.get((name, t), elbow_k)
            assert best_k in K_VALUES
            results[(name, t)] = dict(z=z, models=models, k=best_k, model=models[best_k],
                                      silhouettes=silhouettes)
            comparison_rows.append([name, t, z.shape[1], elbow_k, silhouette_k,
                                     best_k, silhouettes[best_k].mean()])
            axes[0, col].plot([1] + K_VALUES, inertias, 'o-')
            axes[0, col].axvline(elbow_k, color='red', linestyle='--', label='Cot estimat')
            axes[0, col].set(title=f'{t:.0%} varianta', xlabel='k', ylabel='Inertie')
            axes[0, col].legend()
            axes[1, col].plot(K_VALUES, [silhouettes[k].mean() for k in K_VALUES], 'o-')
            axes[1, col].axvline(best_k, color='red', linestyle='--')
            axes[1, col].set(xlabel='k', ylabel='Silhouette mediu')
        fig.suptitle(name)
        plt.tight_layout()
        plt.show()
comparison = pd.DataFrame(comparison_rows, columns=[
    'varianta', 'prag', 'componente', 'k_cot', 'k_silhouette', 'k_ales', 'silhouette'])
display(comparison.round(4))
pd.DataFrame(metric_rows, columns=['varianta', 'prag', 'k', 'inertie', 'silhouette']).to_csv(
    OUTPUT_DIR / 'metrici_k.csv', index=False)


In [ ]:
# Afisam distributia silhouette pentru mai multe valori ale lui k.
for (name, t), result in results.items():
    candidates = sorted(set([2, 3, 4, result['k']]))
    fig, axes = plt.subplots(1, len(candidates), figsize=(4*len(candidates), 4), squeeze=False)
    for ax, k in zip(axes[0], candidates):
        labels = result['models'][k].labels_
        values = result['silhouettes'][k]
        bottom = 0
        for group in range(k):
            v = np.sort(values[labels == group])
            top = bottom + len(v)
            ax.fill_betweenx(np.arange(bottom, top), 0, v, alpha=.7)
            ax.text(-.08, (bottom+top)/2, str(group))
            bottom = top + 10
        ax.axvline(values.mean(), color='red', linestyle='--')
        ax.set(title=f'k={k}, medie={values.mean():.3f}', xlabel='Silhouette',
               ylabel='Case grupate pe clustere', xlim=(-1, 1))
    fig.suptitle(f'{name}, varianta pastrata {t:.0%}')
    plt.tight_layout()
    plt.show()


## 4. Vizualizarea clusterelor în primele două componente principale

Modelul este antrenat în toate dimensiunile păstrate. În scatter plot vedem
doar PC1 și PC2, iar simbolul **X** reprezintă centroidul proiectat pe aceleași axe.
Dacă o variantă păstrează o singură componentă, a doua coordonată este 0.
Suprapunerea în 2D nu demonstrează automat suprapunere în întregul spațiu.

Comparam și **Adjusted Rand Index (ARI)** cu varianta de referință min-max/98%.
ARI=1 înseamnă aceleași apartenențe, chiar dacă numerele clusterelor sunt
permutate. Valori apropiate de 0 indică acord apropiat de cel aleator.


In [ ]:
for name, exp in experiments.items():
    fig, axes = plt.subplots(1, 4, figsize=(19, 4))
    for ax, t in zip(axes, THRESHOLDS):
        r = results[(name, t)]
        z, model = r['z'], r['model']
        y = z[:, 1] if z.shape[1] > 1 else np.zeros(len(z))
        centers = model.cluster_centers_
        cy = centers[:, 1] if z.shape[1] > 1 else np.zeros(len(centers))
        ax.scatter(z[:, 0], y, c=model.labels_, cmap='tab10', s=10, alpha=.5)
        ax.scatter(centers[:, 0], cy, c='black', marker='X', s=140, label='Centroizi')
        ax.set(title=f'{t:.0%}: {z.shape[1]} PC, k={r["k"]}', xlabel='PC1', ylabel='PC2')
        ax.legend()
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()
reference = results[('minmax', .98)]['model'].labels_
comparison['ARI_fata_de_minmax_98'] = [
    adjusted_rand_score(reference, results[(row.varianta, row.prag)]['model'].labels_)
    for row in comparison.itertuples()]
display(comparison.round(4))
comparison.to_csv(OUTPUT_DIR / 'comparatie_clustering.csv', index=False)


### De ce apar diferențe între variante?

Min-max schimbă ponderea fiecărui atribut în distanțe. Normalizarea L2 schimbă
lungimea întregului vector al unei case și poate reduce informația despre
dimensiunea absolută. `log1p` comprimă valorile mari și acceptă valoarea 0.
Prin urmare, toate aceste alegeri modifică ce înseamnă „două case apropiate”.
Reducerea mai puternică prin PCA poate elimina zgomot, dar și diferențe utile
între grupuri. O varianță păstrată mai mare nu garantează un silhouette mai bun.
Comparăm valorile efectiv calculate mai sus; numerele clusterelor nu au
aceeași semnificație de la o variantă la alta.


## 5. Ce atribute influențează componentele principale?

Folosim formula din cerință:

`loadings = pca.components_.T * np.sqrt(pca.explained_variance_)`

O valoare absolută mare indică o contribuție mai puternică în reprezentarea
transformată. Semnul arată direcția în cadrul acelei componente; semnul întregii
componente se poate inversa fără să schimbe interpretarea geometrică.
Aceste valori nu sunt automat corelații cu atributele brute și nici importanțe
predictive precum cele din XGBoost. Heatmap-ul arată primele cinci componente
și 18 atribute cu contribuții mari; exportăm și toate valorile.


In [ ]:
all_loadings = []
for name, exp in experiments.items():
    pca = exp['pca']
    names = exp['prep'].get_feature_names_out()
    count = min(5, len(pca.components_))
    loadings = pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_),
                            index=names, columns=[f'PC{i+1}' for i in range(len(pca.components_))])
    exp['loadings'] = loadings
    all_loadings.append(loadings.assign(varianta=name).rename_axis('atribut').reset_index())
    top = loadings.iloc[:, :count].abs().max(axis=1).nlargest(18).index
    plt.figure(figsize=(9, 8))
    sns.heatmap(loadings.loc[top].iloc[:, :count], cmap='coolwarm', center=0)
    plt.title(f'Contributiile atributelor la componentele PCA: {name}')
    plt.tight_layout()
    plt.show()
    for pc in range(min(3, count)):
        print(name, f'PC{pc+1}:', ', '.join(loadings.iloc[:, pc].abs().nlargest(5).index))
pd.concat(all_loadings).to_csv(OUTPUT_DIR / 'loadings.csv', index=False)


## 6. Ce fel de case găsim în fiecare cluster?

Revenim la **atributele originale**, exprimate în unități ușor de înțeles.
Pentru toate cele 16 modele afișăm numărul caselor și medianele suprafeței
locuibile, terenului, calității, anului construcției și prețului, plus cartierul
cel mai frecvent. Mediana descrie o casă tipică și este mai puțin sensibilă
la valori extreme decât media.

Graficele detaliate folosesc min-max/98%. Prețul este adăugat numai acum,
după clustering. Diferențele observate sunt asocieri descriptive, nu relații cauzale.


In [ ]:
PROFILE_COLS = ['Gr Liv Area', 'Lot Area', 'Overall Qual', 'Year Built', 'SalePrice']
profiles = {}
profile_tables = []
for key, r in results.items():
    grouped = df.assign(cluster=r['model'].labels_).groupby('cluster')
    summary = grouped[PROFILE_COLS].median().round(1)
    summary['numar_case'] = grouped.size()
    summary['cartier_frecvent'] = grouped['Neighborhood'].agg(lambda s: s.mode().iloc[0])
    profiles[key] = summary
    profile_tables.append(summary.assign(varianta=key[0], prag=key[1]).reset_index())
    print(key)
    display(summary)
    for group, row in summary.iterrows():
        print(f'Cluster {group}: {row["numar_case"]} case, suprafata locuibila mediana '
              f'{row["Gr Liv Area"]:.0f} sq ft, calitate {row["Overall Qual"]:.1f}, '
              f'an {row["Year Built"]:.0f}, pret ${row["SalePrice"]:,.0f}; '
              f'cartier frecvent: {row["cartier_frecvent"]}.')
PROFILE_KEY = ('minmax', .98)
profile = df.assign(cluster=results[PROFILE_KEY]['model'].labels_)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, feature in zip(axes, ['Gr Liv Area', 'Overall Qual', 'Year Built', 'SalePrice']):
    sns.boxplot(data=profile, x='cluster', y=feature, ax=ax)
plt.tight_layout()
plt.show()
plt.figure(figsize=(12, 5))
sns.heatmap(pd.crosstab(profile['cluster'], profile['Neighborhood'], normalize='index'), cmap='Blues')
plt.title('Proportia cartierelor in interiorul fiecarui cluster')
plt.tight_layout()
plt.show()
pd.concat(profile_tables).to_csv(OUTPUT_DIR / 'profiluri_clustere.csv', index=False)


## 7. Case inventate și inferență

Inventăm trei exemple pornind de la șabloane complete de case reale, alese
după suprafață. Modificăm suprafețele și calitatea, păstrând celelalte câmpuri
pentru a nu inventa incoerent zeci de categorii. Suprafața locuibilă este
recalculată ca suma etajelor și a suprafeței finisate de calitate redusă.
Acestea sunt variante sintetice noi, nu observații independente de test.

Aplicăm **aceleași** reguli de curățare, imputare, encoding, scalare și PCA
deja învățate. Folosim doar `transform`, niciodată `fit` pe casele noi.
Centroizii sunt în coordonate PCA; o casă exprimată în sq ft și ani nu se
poate compara direct cu ei. Refacerea PCA ar schimba axele, deci și sensul coordonatelor.

Comparăm fiecare casă cu medianele clusterelor. O atribuire surprinzătoare
poate apărea deoarece modelul folosește toate atributele, nu doar calitatea
și suprafața pe care le observăm imediat.


In [ ]:
# Pornim de la sabloane mici, medii si mari, apoi modificam masuratorile.
ordered = X_raw.sort_values('Gr Liv Area')
new_houses = ordered.iloc[[int((len(ordered)-1)*q) for q in [.2, .5, .8]]].copy()
new_houses.index = ['Casa modesta', 'Casa medie', 'Casa mare']
for label, factor, quality in zip(new_houses.index, [.92, 1.03, 1.10], [4, 6, 8]):
    for col in ['1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF']:
        new_houses.loc[label, col] = round(new_houses.loc[label, col] * factor)
    new_houses.loc[label, 'Gr Liv Area'] = new_houses.loc[
        label, ['1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF']].sum()
    new_houses.loc[label, 'Overall Qual'] = quality
    new_houses.loc[label, 'Lot Area'] = round(new_houses.loc[label, 'Lot Area'] * factor)
display(new_houses[['Gr Liv Area', 'Lot Area', 'Overall Qual', 'Year Built', 'Neighborhood']])
assignments = pd.DataFrame(index=new_houses.index)
for (name, t), r in results.items():
    exp = experiments[name]
    encoded = exp['prep'].transform(before_encoding(new_houses, exp['use_log']))
    scaled = exp['scaler'].transform(encoded)
    z_new = exp['pca'].transform(scaled)[:, :exp['counts'][t]]
    assignments[f'{name} / {t:.0%}'] = r['model'].predict(z_new)
display(assignments)

# Comparam aceleasi atribute in casa inventata si in clusterul atribuit.
labels = assignments[f'{PROFILE_KEY[0]} / {PROFILE_KEY[1]:.0%}']
medians = profiles[PROFILE_KEY]
for house, group in labels.items():
    print(f'{house} -> cluster {group}')
    display(pd.DataFrame({'casa_inventata': new_houses.loc[house, PROFILE_COLS[:-1]],
                          'mediana_clusterului': medians.loc[group, PROFILE_COLS[:-1]]}))
assignments.to_csv(OUTPUT_DIR / 'predictii_case_noi.csv')
new_houses.to_csv(OUTPUT_DIR / 'case_noi.csv')


## 8. Bonus: grafice 3D interactive și cele șase clase de preț

Construim două vizualizări în aceleași coordonate PC1–PC3 pentru fiecare
clustering: una colorată după cluster, alta după clasele din L2:
`<100k`, `[100k,130k)`, `[130k,180k)`, `[180k,250k)`, `[250k,400k)`, `>=400k`.
Clasa cea mai scumpă este albastră, iar penultima verde. Centroizii sunt
romburi negre în ambele vizualizări. Meniul permite schimbarea reprezentării;
graficul poate fi rotit și mărit și este salvat ca HTML care funcționează offline.

Dacă PCA a păstrat mai puțin de trei componente, coordonatele lipsă sunt 0.
Nu introducem PC-uri suplimentare care nu au participat la clustering.
Un nor aplatizat în acest caz reflectă reducerea, nu o structură 3D descoperită.
Comparăm și un tabel cluster/clasă de preț și ARI pentru fiecare model.
Clasele de preț nu sunt „adevărul” despre clustere: reprezintă o altă împărțire,
bazată pe o singură variabilă, absentă din antrenarea nesupervizată.


In [ ]:
import plotly.graph_objects as go
from plotly.colors import qualitative

PRICE_NAMES = ['0: <100k', '1: 100k-130k', '2: 130k-180k',
               '3: 180k-250k', '4: 250k-400k', '5: >=400k']
PRICE_COLORS = ['#a6a6a6', '#e78ac3', '#ff9933', '#a6761d', '#009933', '#0066ff']
price_class = pd.cut(df['SalePrice'],
    bins=[-np.inf, 100000, 130000, 180000, 250000, 400000, np.inf],
    labels=False, right=False)

def first_three(values):
    points = np.zeros((len(values), 3))
    count = min(3, values.shape[1])
    points[:, :count] = values[:, :count]
    return points

fig = go.Figure()
views = []
price_rows = []
for (name, t), r in results.items():
    z = first_three(r['z'])
    centers = first_three(r['model'].cluster_centers_)
    cluster_labels = r['model'].labels_
    price_rows.append([name, t, adjusted_rand_score(price_class, cluster_labels)])
    for mode in ['clustere', 'clase de pret']:
        start = len(fig.data)
        color_labels = cluster_labels if mode == 'clustere' else price_class.to_numpy()
        groups = range(r['k']) if mode == 'clustere' else range(6)
        for group in groups:
            mask = color_labels == group
            color = qualitative.Plotly[group % 10] if mode == 'clustere' else PRICE_COLORS[group]
            label = f'Cluster {group}' if mode == 'clustere' else PRICE_NAMES[group]
            fig.add_trace(go.Scatter3d(
                x=z[mask, 0], y=z[mask, 1], z=z[mask, 2], mode='markers', name=label,
                marker=dict(size=3, color=color, opacity=.65), visible=False,
                customdata=np.column_stack([df.loc[mask, 'Gr Liv Area'],
                                             df.loc[mask, 'SalePrice']]),
                hovertemplate='Suprafata: %{customdata[0]} sq ft<br>Pret: $%{customdata[1]:,.0f}<extra>%{fullData.name}</extra>'))
        fig.add_trace(go.Scatter3d(
            x=centers[:, 0], y=centers[:, 1], z=centers[:, 2], mode='markers+text',
            marker=dict(size=8, color='black', symbol='diamond'),
            text=[f'C{i}' for i in range(r['k'])], name='Centroizi', visible=False))
        title = f'{name}, varianta {t:.0%}, {r["z"].shape[1]} PC, k={r["k"]}: {mode}'
        views.append((title, start, len(fig.data)))
buttons = []
for title, start, end in views:
    visible = [start <= i < end for i in range(len(fig.data))]
    buttons.append(dict(label=title, method='update', args=[{'visible': visible}, {'title': title}]))
for i in range(views[0][1], views[0][2]):
    fig.data[i].visible = True
fig.update_layout(title=views[0][0], height=720,
                  scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'),
                  updatemenus=[dict(buttons=buttons, x=0, y=1.15)], margin=dict(t=160))
fig.write_html(OUTPUT_DIR / 'clustere_3d_interactiv.html', include_plotlyjs=True)
display(fig)

price_comparison = pd.DataFrame(price_rows, columns=['varianta', 'prag', 'ARI_clase_pret'])
display(price_comparison.round(4))
price_comparison.to_csv(OUTPUT_DIR / 'comparatie_clase_pret.csv', index=False)
fig2, axes = plt.subplots(1, 2, figsize=(14, 4))
counts_price = pd.crosstab(profile['cluster'], price_class).reindex(columns=range(6), fill_value=0)
counts_price.columns = PRICE_NAMES
sns.heatmap(counts_price, annot=True, fmt='d', cmap='Blues', ax=axes[0])
sns.heatmap(counts_price.div(counts_price.sum(axis=1), axis=0), annot=True,
            fmt='.1%', cmap='Blues', ax=axes[1])
axes[0].set_title('Numarul caselor pe cluster si clasa de pret')
axes[1].set_title('Proportii in interiorul clusterelor')
for ax in axes:
    ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()
counts_price.to_csv(OUTPUT_DIR / 'cluster_vs_clasa_pret.csv')


### Interpretarea bonusului și limitele K-means

A treia componentă poate separa puncte suprapuse în 2D; verificăm acest lucru
rotind graficul, fără să confundăm proiecția cu întregul spațiu al modelului.
Colorarea după preț arată cât de mult se suprapun clasele economice în
aceleași regiuni. Nu ne așteptăm la egalitate: K-means minimizează distanțe,
în timp ce L2 învață să prezică praguri de preț. În L2, atributele de suprafață
și calitate și combinațiile lor sunt relevante pentru predicție; aici putem
vedea concret cât de mult contribuie ele la PC-uri prin loadings.

K-means este un punct de pornire interpretabil, dar favorizează grupuri
compacte, aproximativ sferice în spațiul ales. Casele au atribute mixte și
variații continue, iar clusterele se pot suprapune. O separare impusă nu
demonstrează existența unor tipuri naturale distincte de case. Pentru grupuri
eliptice am putea compara Gaussian Mixture; pentru forme neregulate și zgomot,
DBSCAN/HDBSCAN; pentru atribute mixte, o distanță Gower cu un algoritm compatibil.
Aceste alternative nu au fost antrenate aici, deci nu pretindem că sunt superioare.


## 9. Concluzii pe baza rezultatelor calculate

Celula următoare sintetizează rezultatele reale ale rulării: numărul de PC-uri,
alegerile lui k, profilurile clusterelor, efectul logaritmării și casele noi.
Interpretarea distanței caselor față de profil este descriptivă, nu un test
al „corectitudinii” unei etichete nesupervizate.


In [ ]:
lines = ['# L3: rezultate calculate pentru Ames Housing', '',
         f'Am pastrat **{len(df)} case** si am eliminat **{int(outliers.sum())} outlieri**.', '',
         '## Numarul componentelor principale', '',
         '| Varianta | 98% | 95% | 90% | 80% |',
         '|---|---:|---:|---:|---:|']
for name, exp in experiments.items():
    values = ' | '.join(str(exp['counts'][t]) for t in THRESHOLDS)
    lines.append(f'| {name} | {values} |')
lines += ['', '## Alegerea clusterelor', '',
          '| Varianta | Prag | k cot | k silhouette | k ales | Silhouette | ARI fata de referinta |',
          '|---|---:|---:|---:|---:|---:|---:|']
for row in comparison.itertuples():
    lines.append(f'| {row.varianta} | {row.prag:.0%} | {row.k_cot} | {row.k_silhouette} | '
                 f'{row.k_ales} | {row.silhouette:.3f} | {row.ARI_fata_de_minmax_98:.3f} |')
lines += ['', 'Cotul este o estimare pe k=1,...,10. Silhouette poate prefera mai putine '
          'grupuri; pastram alegerea cotului ca segmentare exploratorie, fara a pretinde '
          'ca aceasta este unica impartire corecta.', '', '## Efectul preprocesarii', '']
for scaling in ['minmax', 'normalize']:
    a = comparison[(comparison.varianta == scaling) & (comparison.prag == .98)].iloc[0]
    b = comparison[(comparison.varianta == scaling + ' + log1p') & (comparison.prag == .98)].iloc[0]
    agreement = adjusted_rand_score(results[(scaling, .98)]['model'].labels_,
                                    results[(scaling + ' + log1p', .98)]['model'].labels_)
    lines.append(f'- La {scaling}/98%, log1p schimba silhouette de la {a.silhouette:.3f} '
                 f'la {b.silhouette:.3f}, k de la {a.k_ales} la {b.k_ales}, iar acordul '
                 f'dintre apartenente are ARI={agreement:.3f}.')
lines += ['', 'Logaritmarea comprima extremele. Normalizarea L2 directa este sensibila '
          'la unitati: un numar mic de PC-uri poate explica varianta unor masuratori '
          'dominante, fara sa pastreze diversitatea semantica a caselor. Scorurile din '
          'spatii diferite trebuie citite impreuna cu profilurile, nu ca un clasament absolut.',
          '', '## Atributele dominante', '']
for name, exp in experiments.items():
    for pc in ['PC1', 'PC2']:
        top = exp['loadings'][pc].abs().nlargest(5).index
        lines.append(f'- {name}, {pc}: ' + ', '.join(top) + '.')
lines += ['', '## Profilurile clusterelor de referinta: min-max/98%', '']
for group, row in medians.iterrows():
    lines.append(f'- **Cluster {group}**: {row.numar_case} case, suprafata locuibila mediana '
                 f'{row["Gr Liv Area"]:.0f} sq ft, teren {row["Lot Area"]:.0f} sq ft, '
                 f'calitate {row["Overall Qual"]:.1f}/10, an {row["Year Built"]:.0f}, '
                 f'pret ${row.SalePrice:,.0f}, cartier frecvent {row.cartier_frecvent}.')
lines += ['', '## Casele noi: interpretarea atribuirilor', '']
for house, group in labels.items():
    invented = new_houses.loc[house]
    typical = medians.loc[group]
    area_gap = 100 * (invented['Gr Liv Area'] / typical['Gr Liv Area'] - 1)
    quality_gap = invented['Overall Qual'] - typical['Overall Qual']
    lines.append(f'- **{house} → cluster {group}**: {invented["Gr Liv Area"]:.0f} sq ft '
                 f'({area_gap:+.1f}% fata de mediana clusterului), calitate '
                 f'{invented["Overall Qual"]:.0f} ({quality_gap:+.1f} puncte fata de mediana), '
                 f'an {invented["Year Built"]:.0f} fata de {typical["Year Built"]:.0f}.')
    verdict = ('Atribuirea este plauzibila dupa suprafata si calitate.'
               if abs(area_gap) < 25 and abs(quality_gap) <= 1
               else 'Atribuirea este mai putin evidenta dupa suprafata/calitate; '
                    'cartierul si celelalte atribute pot explica apropierea de centroid.')
    lines.append('  ' + verdict)
lines += ['', 'Casele sunt variante sintetice ale unor sabloane reale. Aceste comparatii '
          'explica predictiile, dar nu masoara performanta pe case independente.', '',
          '## Clustere si clase de pret', '']
ref_ari = adjusted_rand_score(price_class, reference)
lines.append(f'ARI dintre clusterele de referinta si cele sase clase din L2 este '
             f'**{ref_ari:.3f}**. Clasele de pret si clusterele nu coincid. '
             'O casa poate avea un pret diferit de alte case cu atribute apropiate; '
             'in plus, PCA pastreaza varianta atributelor, nu separarea pragurilor de pret.')
lines += ['', 'K-means este util pentru o prima segmentare, dar scorurile silhouette si '
          'suprapunerea claselor cer prudenta: grupurile nu reprezinta o clasificare '
          'garantata a valorii unei case. Graficele 3D sunt proiectii ale modelelor '
          'si nu justifica singure alegerea unei metode alternative.', '',
          'Rezultatele depind de versiunea bibliotecilor si de alegerile explicite '
          'de encoding, scalare si estimare a cotului.']
report = '\n'.join(lines) + '\n'
(OUTPUT_DIR / 'REZULTATE.md').write_text(report, encoding='utf-8')
display(Markdown(report))


## Interpretarea rulării de referință

Pe CSV-ul din repository rămân **2.927 de case**, după eliminarea a trei
outlieri. Min-max are 287 de atribute după codificare; PCA păstrează **135,
98, 72, respectiv 45 de componente** pentru pragurile de 98%, 95%, 90%, 80%.

**Cotul nu este foarte pronunțat.** Estimatorul propune k=4 la min-max/98%,
în timp ce silhouette preferă k=3 (0,094 față de 0,074 la k=4). Ambele valori
sunt mici: casele nu formează grupuri puternic separate. Păstrăm patru grupuri
pentru interpretarea exploratorie cerută, fiindcă metoda cotului le sugerează
și profilurile lor sunt distincte. Nu prezentăm această alegere ca optim unic.

La min-max, reducerea de la 98% la 80% schimbă foarte puțin apartenențele
(ARI față de referință rămâne peste 0,99). Logaritmarea schimbă și ea puțin
acest rezultat (ARI=0,982 la 98%), fără să îmbunătățească silhouette.
În schimb, normalizarea L2 directă produce o împărțire foarte diferită
(ARI=0,038 față de min-max/98%). În PC1 domină terenul și anii, ceea ce arată
efectul unităților de măsură. Scorul silhouette mai mare nu este suficient
pentru a declara această reprezentare mai potrivită semantic.

**PC1 la min-max** opune, în orientarea acestei rulări, fundațiile din beton
turnat (`Foundation_PConc`, contribuție pozitivă) și finisajele exterioare din
vinil garajelor nefinisate/detașate (contribuții negative). Axa este astfel
asociată unor diferențe de construcție și dotări. **PC2** diferențiază mai ales
casele cu un nivel (`1Story`, `MS SubClass_20`, semn pozitiv) de cele cu două
niveluri (`2Story`, semn negativ). **PC3** conține de asemenea informație despre
stilul pe niveluri și forma terenului; graficul 3D adaugă această perspectivă,
fără să elimine suprapunerile sau nevoia celorlalte componente.

Profilurile min-max/98% arată patru tipuri descriptive:

- **Cluster 0:** case mai vechi (an median 1931), calitate 5 și preț median
  de 123.000 USD, frecvente în OldTown.
- **Cluster 1:** case recente (2005), calitate 7, suprafață mediană de 1.502
  sq ft și preț median de 221.500 USD, frecvente în CollgCr.
- **Cluster 2:** case recente (2000), mai mari (1.893 sq ft), calitate 7 și
  preț median de 220.000 USD, frecvente în Gilbert.
- **Cluster 3:** case mai compacte (1.211 sq ft), an median 1965 și calitate
  5, cu preț median de 145.000 USD, frecvente în NAmes.

Casa modestă și casa medie sunt atribuite clusterului 3. Pentru prima,
979 sq ft și calitatea 4 sunt apropiate de profilul acelui grup. Casa medie
are 1.485 sq ft, calitate 6 și anul 1990, deci este mai mare și mai nouă decât
mediana clusterului; atribuirea este mai puțin evidentă doar din aceste trei
atribute. Celelalte caracteristici din șablon participă însă și ele la distanță.
Casa mare ajunge în clusterul 2: 2.021 sq ft, calitate 8 și anul 1997 sunt
compatibile cu profilul caselor mari și relativ recente. Numerele clusterelor
sunt arbitrare și nu reprezintă niveluri de preț.

Comparația cu L2 confirmă această diferență: **ARI=0,144** între clusterele
de referință și clasele de preț. De exemplu, în clusterul 3, 54,3% dintre case
au prețuri în `[130k,180k)`, dar există și alte clase. Clusterele 1 și 2 au
medianele prețului foarte apropiate, deși suprafețele lor diferă. PCA este
influențat și de fundație, garaj și stil, nu numai de factorii care separă
pragurile de preț din clasificarea supervizată.

Valorile de aici descriu rularea verificată pe datele curente. Tabelele
generate la Run All sunt autoritative dacă schimbăm datele sau parametrii.


## Referințe și reproducere

- Cerința de laborator: **L3(2).pdf**, „Principal Component Analysis und K-means Clustering”.
- [PCA — documentația scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [Clustering și limitele K-means](https://scikit-learn.org/stable/modules/clustering.html)
- [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)
- [Normalizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.Normalizer.html)

Seed-ul este 42. Calculăm silhouette pe toate casele, nu pe un eșantion.
O rulare poate dura câteva minute. Fișierele HTML/CSV din `rezultate/` pot fi
regenerate prin Run All; raportul verificat este și în `REZULTATE.md`, lângă notebook.
